Не очень понял, честно, как лучше реализовать пункт про оптимизацию и доступность воспроизводимости. Сделал архитектуру сохранения истории обучений. Там есть логи, резы обучения. Есть только единственный наилучший вес. Ноутбук выполнен для наилучшего веса

In [1]:
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / "morse_dataset"
TEST_DIR = DATA_DIR / "test"
CHECKPOINT_DIR = ROOT / "weights"
BEST_PATH = CHECKPOINT_DIR / "best.pt"
METRICS_PATH = CHECKPOINT_DIR / "metrics.csv"
BEAM_SIZE = 10
SUBMISSION_PATH = CHECKPOINT_DIR / f"submission_beam{BEAM_SIZE}.csv"

print("root:", ROOT)
print("test_dir:", TEST_DIR)
print("checkpoint:", BEST_PATH)
print("test wavs:", len(list(TEST_DIR.glob("*.wav"))) if TEST_DIR.exists() else 0)
print("best.pt exists:", BEST_PATH.exists())
print("best.pt size MB:", round(BEST_PATH.stat().st_size / 1024**2, 2) if BEST_PATH.exists() else None)

assert TEST_DIR.exists(), f"Missing test dir: {TEST_DIR}"
assert BEST_PATH.exists(), f"Missing checkpoint: {BEST_PATH}"

root: D:\dev\ctc-contest
test_dir: D:\dev\ctc-contest\morse_dataset\test
checkpoint: D:\dev\ctc-contest\weights\best.pt
test wavs: 5000
best.pt exists: True
best.pt size MB: 38.62


In [2]:
import csv
import math
import random
import wave

import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

CHARS = "-0123456789"
BLANK = 0
char_to_id = {char: index + 1 for index, char in enumerate(CHARS)}
id_to_char = {index + 1: char for index, char in enumerate(CHARS)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
device: cuda
gpu: NVIDIA GeForce RTX 3050


In [3]:
def read_test_rows(test_dir):
    files = sorted(Path(test_dir).glob("*.wav"))
    if not files:
        raise FileNotFoundError()
    print(f"using {len(files)} test wav files")
    return [{"filename": path.name, "text": ""} for path in files]


def read_wav(path):
    with wave.open(str(path), "rb") as file:
        if file.getframerate() != 8000 or file.getnchannels() != 1 or file.getsampwidth() != 2:
            raise ValueError()
        raw = bytearray(file.readframes(file.getnframes()))

    return torch.frombuffer(raw, dtype=torch.int16).float() / 32768.0


class MorseDataset(Dataset):
    def __init__(self, rows, audio_dir):
        self.rows = rows
        self.audio_dir = Path(audio_dir)
        self.window = torch.hann_window(256)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        wav = read_wav(self.audio_dir / row["filename"])
        spec = torch.stft(wav, n_fft=256, hop_length=80, win_length=256, window=self.window, center=False, return_complex=True).abs()
        spec = torch.log1p(spec).transpose(0, 1)
        spec = (spec - spec.mean()) / (spec.std() + 1e-5)
        return spec, None, row["filename"]


def collate(batch):
    specs, targets, filenames = zip(*batch)
    input_lengths = torch.tensor([len(spec) for spec in specs], dtype=torch.long)
    specs = pad_sequence(specs, batch_first=True)
    return specs, input_lengths, None, None, filenames

In [4]:
class MorseCTC(nn.Module):
    def __init__(self):
        super().__init__()
        hidden_size = 256
        self.conv = nn.Sequential(
            nn.Conv1d(129, 192, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(192),
            nn.ReLU(),
            nn.Conv1d(192, 192, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(192),
            nn.ReLU(),
        )
        self.rnn = nn.GRU(input_size=192, hidden_size=hidden_size, num_layers=3, dropout=0.2, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(hidden_size * 2, len(CHARS) + 1)

    def forward(self, specs, lengths):
        x = self.conv(specs.transpose(1, 2)).transpose(1, 2)
        lengths = (lengths + 1) // 2
        lengths = (lengths + 1) // 2
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed, _ = self.rnn(packed)
        x, _ = pad_packed_sequence(packed, batch_first=True)
        return self.classifier(x).log_softmax(dim=-1).transpose(0, 1), lengths


def load_model(checkpoint_path, device):
    model = MorseCTC().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if checkpoint.get("chars") != CHARS:
        raise ValueError()
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model, checkpoint

In [5]:
def greedy_decode(log_probs, lengths):
    best = log_probs.argmax(dim=-1).transpose(0, 1).cpu()
    answers = []

    for ids, length in zip(best, lengths):
        previous = BLANK
        text = []

        for token in ids[: int(length)]:
            token = int(token)
            if token != BLANK and token != previous:
                text.append(id_to_char[token])

            previous = token
        answers.append("".join(text))

    return answers


def logsumexp_pair(left, right):
    if left == -math.inf:
        return right

    if right == -math.inf:
        return left

    if left < right:
        left, right = right, left

    return left + math.log1p(math.exp(right - left))


def ctc_beam_decode_one(sequence, beam_size):
    beams = {"": (0.0, -math.inf)}

    for frame in sequence:
        next_beams = {}
        top_ids = torch.topk(frame, k=min(beam_size, frame.numel())).indices.tolist()

        for prefix, (blank_score, nonblank_score) in beams.items():
            total_score = logsumexp_pair(blank_score, nonblank_score)

            for token in top_ids:
                score = float(frame[token])
                if token == BLANK:
                    old_blank, old_nonblank = next_beams.get(prefix, (-math.inf, -math.inf))
                    old_blank = logsumexp_pair(old_blank, total_score + score)
                    next_beams[prefix] = (old_blank, old_nonblank)
                    continue

                char = id_to_char[token]
                if prefix and char == prefix[-1]:
                    old_blank, old_nonblank = next_beams.get(prefix, (-math.inf, -math.inf))
                    old_nonblank = logsumexp_pair(old_nonblank, nonblank_score + score)
                    next_beams[prefix] = (old_blank, old_nonblank)
                    extended = prefix + char
                    old_blank, old_nonblank = next_beams.get(extended, (-math.inf, -math.inf))
                    old_nonblank = logsumexp_pair(old_nonblank, blank_score + score)
                    next_beams[extended] = (old_blank, old_nonblank)

                else:
                    extended = prefix + char
                    old_blank, old_nonblank = next_beams.get(extended, (-math.inf, -math.inf))
                    old_nonblank = logsumexp_pair(old_nonblank, total_score + score)
                    next_beams[extended] = (old_blank, old_nonblank)

        beams = dict(sorted(next_beams.items(),
                            key=lambda item: logsumexp_pair(item[1][0], item[1][1]),
                            reverse=True)[:beam_size])

    return max(beams.items(),
               key=lambda item: logsumexp_pair(item[1][0], item[1][1]))[0]


def beam_decode(log_probs, lengths, beam_size):
    batch = log_probs.transpose(0, 1).cpu()
    return [ctc_beam_decode_one(batch[index, : int(length)], beam_size) for index, length in enumerate(lengths)]


def decode_batch(log_probs, lengths, beam_size):
    if beam_size <= 1:
        return greedy_decode(log_probs, lengths)
    return beam_decode(log_probs, lengths, beam_size)

In [6]:
@torch.inference_mode()
def predict(checkpoint_dir, data_dir, submission_path, beam_size=10, predict_batch_size=16, workers=0, log_every=50):
    checkpoint_dir = Path(checkpoint_dir)
    data_dir = Path(data_dir)
    submission_path = Path(submission_path)
    rows = read_test_rows(data_dir / "test")
    dataset = MorseDataset(rows, data_dir / "test")

    loader = DataLoader(dataset,
                        batch_size=predict_batch_size,
                        shuffle=False,
                        num_workers=workers,
                        collate_fn=collate,
                        pin_memory=torch.cuda.is_available())

    model, checkpoint = load_model(checkpoint_dir / "best.pt", device)
    predictions = []

    for step, (specs, input_lengths, _, _, _) in enumerate(loader, start=1):
        log_probs, output_lengths = model(specs.to(device, non_blocking=True), input_lengths)
        predictions.extend(decode_batch(log_probs, output_lengths, beam_size))

        if step % log_every == 0:
            print(f"predict step={step}/{len(loader)}")

    with open(submission_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["filename", "text"])
        writer.writeheader()

        for row, text in zip(rows, predictions):
            writer.writerow({"filename": row["filename"], "text": text})

    print(f"saved={submission_path} rows={len(predictions)}")
    return submission_path

In [7]:
submission_path = predict(
    checkpoint_dir=CHECKPOINT_DIR,
    data_dir=DATA_DIR,
    submission_path=SUBMISSION_PATH,
    beam_size=BEAM_SIZE,
    predict_batch_size=16,
    workers=0,
    log_every=50,
)
print("submission:", submission_path)

using 5000 test wav files


predict step=50/313


predict step=100/313


predict step=150/313


predict step=200/313


predict step=250/313


predict step=300/313


saved=D:\dev\ctc-contest\weights\submission_beam10.csv rows=5000
submission: D:\dev\ctc-contest\weights\submission_beam10.csv


In [8]:
submission = pd.read_csv(SUBMISSION_PATH)
print("shape:", submission.shape)
submission.head(10)

shape: (5000, 2)


,filename,text
0,00000.wav,964-69-44
1,00001.wav,00-4
2,00002.wav,668
3,00003.wav,4481-1180
4,00004.wav,5-4
5,00005.wav,000
6,00006.wav,00033
7,00007.wav,84
8,00008.wav,343
9,00009.wav,17


In [9]:
if METRICS_PATH.exists():
    metrics = pd.read_csv(METRICS_PATH)
    best_row = metrics.loc[metrics["val_levenshtein"].idxmin()]
    print("best epoch:", int(best_row["epoch"]))
    print("best val_levenshtein:", float(best_row["val_levenshtein"]))
    display(metrics.tail())
else:
    print("metrics.csv is optional for inference and was not found")

best epoch: 19
best val_levenshtein: 0.170667


,epoch,train_loss,val_levenshtein,best_val
15,16,0.080964,0.205000,0.186667
16,17,0.080157,0.182333,0.182333
17,18,0.075404,0.184333,0.182333
18,19,0.072815,0.170667,0.170667
19,20,0.072489,0.207333,0.170667
